# D531 GCP CLI on Ubuntu

Install Google Cloud CLI, configure a project, and verify access to Google Cloud Storage (GCS). Use the existing **dataengenv** for Python libraries.

Basic AWS CLI knowledge is assumed; the commands below map the same tasks to GCP.

**Run every Bash block in an Ubuntu terminal, in order, using the same terminal session.** These are terminal instructions, not Python notebook cells. You need sudo access for installation and a Google account with access to a GCP project.

## 1. Activate dataengenv

Use the existing environment. The example assumes it is at `~/dataengenv`; adjust the path if it is elsewhere.

```bash
source ~/dataengenv/bin/activate
python --version
python -m pip --version
which python
```

The Python path should point inside `dataengenv`.

**Two installations:** `apt` installs the Google Cloud CLI for Ubuntu. `pip` installs Python client libraries inside `dataengenv`. Activating or deactivating the environment does not install or remove `gcloud`.

## 2. Install Google Cloud CLI

Install prerequisites:

```bash
sudo apt-get update
sudo apt-get install -y ca-certificates gnupg curl
```

Download and import the repository key. Continue only if each command succeeds:

```bash
curl -fsSL https://packages.cloud.google.com/apt/doc/apt-key.gpg -o /tmp/google-cloud-apt-key.gpg
sudo gpg --dearmor --yes --output /usr/share/keyrings/cloud.google.gpg /tmp/google-cloud-apt-key.gpg
```

Register the package repository:

```bash
echo "deb [signed-by=/usr/share/keyrings/cloud.google.gpg] https://packages.cloud.google.com/apt cloud-sdk main" | sudo tee /etc/apt/sources.list.d/google-cloud-sdk.list
```

Install and verify:

```bash
sudo apt-get update
sudo apt-get install -y google-cloud-cli
gcloud --version
```

Expected: version information. GCS commands are included as `gcloud storage`; no separate GCS CLI installation is needed.

[Reference: Ubuntu installation](https://docs.cloud.google.com/sdk/docs/install-sdk#deb)

## 3. Create a CLI configuration and sign in

A named gcloud configuration is similar in purpose to an AWS CLI profile.

```bash
gcloud config configurations create training
```

Creation activates it. If `training` already exists, use this instead:

```bash
gcloud config configurations activate training
```

Sign in using a browser:

```bash
gcloud auth login
```

For an Ubuntu machine without a usable local browser, use this alternative and follow the displayed instructions:

```bash
gcloud auth login --no-launch-browser
```

Check the selected account:

```bash
gcloud auth list
```

The active account has an asterisk. Run authentication commands as your normal Ubuntu user, without `sudo`.

[Reference: CLI authentication](https://docs.cloud.google.com/sdk/docs/authenticate)

## 4. Select the project

Replace the placeholder with the actual **project ID**, not the display name.

```bash
export GOOGLE_CLOUD_PROJECT="YOUR_PROJECT_ID"
gcloud config set project "$GOOGLE_CLOUD_PROJECT"
gcloud config list
```

The export also supplies a project to Python client libraries launched from this terminal. It lasts for this terminal session; repeat it in a new terminal.

A GCS listing does not need a default compute region. If later exercises use Compute Engine or Dataproc, set the relevant service location then; `compute/region` does not set a bucket's location.

## 5. Verify GCS access with the CLI

List buckets in the selected project:

```bash
gcloud storage buckets list --project="$GOOGLE_CLOUD_PROJECT"
```

Expected: bucket details, or an empty result if the project has no buckets. The account needs `storage.buckets.list` on the project.

If an existing bucket is available, replace the placeholder and list its objects:

```bash
gcloud storage ls gs://YOUR_EXISTING_BUCKET/
```

Listing objects requires `storage.objects.list` on that bucket. Successful login does not automatically grant storage permissions.

[Reference: Listing buckets](https://docs.cloud.google.com/storage/docs/listing-buckets)

## 6. Configure Python authentication

`gcloud auth login` authenticates CLI commands. Python client libraries use **Application Default Credentials (ADC)**.

```bash
gcloud auth application-default login
```

On a machine without a usable local browser, use this alternative:

```bash
gcloud auth application-default login --no-launch-browser
```

Use the same Google account for this exercise. ADC is separate from named CLI configurations; switching configurations does not switch ADC credentials.

If a quota-project warning appears, set an authorized project:

```bash
gcloud auth application-default set-quota-project "$GOOGLE_CLOUD_PROJECT"
```

This requires `serviceusage.services.use` on that project. No access-token printing is needed for verification.

[Reference: Local ADC setup](https://docs.cloud.google.com/docs/authentication/set-up-adc-local-dev-environment)

## 7. Install the GCS Python library in dataengenv

Confirm the environment is active, then install:

```bash
which python
python -m pip install google-cloud-storage
python -m pip show google-cloud-storage
```

The package location should be inside `dataengenv`. It also installs its required authentication dependencies.

Use `python -m pip` so installation targets the same Python interpreter used for the next check.

## 8. Verify Python access to GCS

Paste this entire block into the same Ubuntu terminal. It runs Python from `dataengenv` and uses ADC.

```bash
python - <<'PY'
import os
import google.auth
from google.auth.transport.requests import Request
from google.cloud import storage

project_id = os.environ["GOOGLE_CLOUD_PROJECT"]
credentials, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
credentials.refresh(Request())
print("Authentication succeeded.")
print("Project:", project_id)

client = storage.Client(project=project_id, credentials=credentials)
buckets = list(client.list_buckets(max_results=5))
for bucket in buckets:
    print(bucket.name)
print("GCS access succeeded; buckets shown:", len(buckets))
PY
```

Expected: authentication succeeds, followed by up to five bucket names. Zero buckets is a valid result. A permission error after authentication means the account or project permissions need checking.

This check reads bucket metadata and does not create resources.

## 9. AWS command mapping

| Task | AWS | GCP |
|---|---|---|
| CLI command | `aws` | `gcloud` |
| Named setup | AWS profile | gcloud configuration |
| Select setup for one command | `--profile training` | `--configuration=training` |
| List storage buckets | `aws s3 ls` | `gcloud storage buckets list` |
| List objects | `aws s3 ls s3://BUCKET/` | `gcloud storage ls gs://BUCKET/` |
| Python storage client | `boto3` | `google-cloud-storage` |

List or activate configurations:

```bash
gcloud config configurations list
gcloud config configurations activate training
```

Use a configuration for one command:

```bash
gcloud storage buckets list --configuration=training
```

[Reference: gcloud configurations](https://docs.cloud.google.com/sdk/docs/configurations)

## 10. Quick fixes

| Problem | Check or action |
|---|---|
| `dataengenv/bin/activate` not found | Use the actual path of the existing environment. |
| `gcloud: command not found` | Confirm the apt installation completed successfully; run `command -v gcloud`. |
| Repository reports conflicting `Signed-By` values | Check for duplicate Google Cloud repository entries before retrying apt. |
| Wrong CLI account or project | Run `gcloud auth list` and `gcloud config list`. |
| CLI works but Python cannot find credentials | Complete the separate ADC login as the same Ubuntu user running Python. |
| Python reports `ModuleNotFoundError` | Activate `dataengenv` and repeat `python -m pip install google-cloud-storage`. |
| GCS returns `403` | Check the selected project and the account's bucket-listing or object-listing permissions. |
| Python uses unexpected credentials | Check whether `GOOGLE_APPLICATION_CREDENTIALS` points to another credential file; it takes precedence over local ADC. |

**Setup complete:** `gcloud --version` works, the intended project is selected, and GCS access works from the CLI and Python.